### Step 1: Import Required Libraries
Import the required Spark libraries and create a Spark Session.

In [ ]:
# Import required Spark libraries
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [ ]:
# Create a Spark session
spark = SparkSession.builder.appName("W5Assignment").getOrCreate()

In [3]:
print(spark)

### Step 2: Load and Explore Dataset
Load the CSV dataset into a Spark DataFrame and perform basic exploration.

In [4]:
# Load the CSV dataset
df = spark.read.csv("dataset.csv", header=True, inferSchema=True)

In [5]:
# Display first 5 records
df.show(5)

+-------+----------------+----------------+-----------+------+------+---+------------+---------+-------+--------+-------------------+-----------------+--------+
|user_id|transaction_date|product_category|sale_amount|region|  city|age|subscription|   status|  price|store_id|      raw_timestamp|            email|username|
+-------+----------------+----------------+-----------+------+------+---+------------+---------+-------+--------+-------------------+-----------------+--------+
|  U1001|      2026-06-27|       Groceries|    3453.62|  East|Jaipur| 37|     Premium|Cancelled|2558.76|    S104|2026-01-01 20:22:00|user1@example.com|   user1|
|  U1002|      2026-02-18|     Electronics|    4759.89|  West|  Pune| 47|     Premium|Completed| 113.78|    S103|2026-05-24 06:27:00|user2@example.com|   user2|
|  U1003|      2026-03-29|        Clothing|    1567.15| South|Mumbai| 31|       Basic|Cancelled|2052.65|    S105|2026-01-09 07:53:00|user3@example.com|   user3|
|  U1004|      2026-05-21|        

In [6]:
# Display dataset schema
df.printSchema()

root
 |-- user_id: string (nullable = true)
 |-- transaction_date: date (nullable = true)
 |-- product_category: string (nullable = true)
 |-- sale_amount: double (nullable = true)
 |-- region: string (nullable = true)
 |-- city: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- subscription: string (nullable = true)
 |-- status: string (nullable = true)
 |-- price: double (nullable = true)
 |-- store_id: string (nullable = true)
 |-- raw_timestamp: string (nullable = true)
 |-- email: string (nullable = true)
 |-- username: string (nullable = true)



In [7]:
# Count total number of rows
df.count()

500

In [8]:
# Display column names and total columns
print("Columns:" , df.columns)
print("Number of columns:", len(df.columns))

Columns: ['user_id', 'transaction_date', 'product_category', 'sale_amount', 'region', 'city', 'age', 'subscription', 'status', 'price', 'store_id', 'raw_timestamp', 'email', 'username']
Number of columns: 14


### Q3 - Remove Duplicate Records
Remove duplicate rows based on the `user_id` and `transaction_date` columns.

In [9]:
# Remove duplicates based on user_id and transaction_date

df_no_duplicates = df.dropDuplicates(["user_id", "transaction_date"])

print("Rows before removing duplicates:", df.count())
print("Rows after removing duplicates:", df_no_duplicates.count())

df_no_duplicates.show(5)

Rows before removing duplicates: 500
Rows after removing duplicates: 480
+-------+----------------+----------------+-----------+------+------+---+------------+---------+-------+--------+-------------------+-----------------+--------+
|user_id|transaction_date|product_category|sale_amount|region|  city|age|subscription|   status|  price|store_id|      raw_timestamp|            email|username|
+-------+----------------+----------------+-----------+------+------+---+------------+---------+-------+--------+-------------------+-----------------+--------+
|  U1001|      2026-06-27|       Groceries|    3453.62|  East|Jaipur| 37|     Premium|Cancelled|2558.76|    S104|2026-01-01 20:22:00|user1@example.com|   user1|
|  U1002|      2026-02-18|     Electronics|    4759.89|  West|  Pune| 47|     Premium|Completed| 113.78|    S103|2026-05-24 06:27:00|user2@example.com|   user2|
|  U1003|      2026-03-29|        Clothing|    1567.15| South|Mumbai| 31|       Basic|Cancelled|2052.65|    S105|2026-01-0

In [10]:
# Use the updated DataFrame
df = df_no_duplicates

### Q4 - Filter and Aggregate Data
Filter records where the region is **West** and calculate the average sale amount for each product category.

In [11]:
# Filter records where region is 'West'
west_sales = df.filter(col("region") == "West")

# Group by product category and calculate average sale amount
avg_sales = west_sales.groupBy("product_category").agg(avg("sale_amount").alias("average_sale_amount"))

avg_sales.show()

+----------------+-------------------+
|product_category|average_sale_amount|
+----------------+-------------------+
|          Sports| 2820.9434615384607|
|       Groceries| 3186.9576666666662|
|     Electronics| 2322.0270000000005|
|        Clothing| 2234.2345454545457|
|       Furniture| 2373.1550000000007|
+----------------+-------------------+



### Q5 - Handle Missing Values
Demonstrate the use of `.na.drop()` and `.na.fill()` to handle missing values.

In [12]:
# Removing rows with null values
df.na.drop().show(5)

+-------+----------------+----------------+-----------+------+------+---+------------+---------+-------+--------+-------------------+-----------------+--------+
|user_id|transaction_date|product_category|sale_amount|region|  city|age|subscription|   status|  price|store_id|      raw_timestamp|            email|username|
+-------+----------------+----------------+-----------+------+------+---+------------+---------+-------+--------+-------------------+-----------------+--------+
|  U1001|      2026-06-27|       Groceries|    3453.62|  East|Jaipur| 37|     Premium|Cancelled|2558.76|    S104|2026-01-01 20:22:00|user1@example.com|   user1|
|  U1002|      2026-02-18|     Electronics|    4759.89|  West|  Pune| 47|     Premium|Completed| 113.78|    S103|2026-05-24 06:27:00|user2@example.com|   user2|
|  U1003|      2026-03-29|        Clothing|    1567.15| South|Mumbai| 31|       Basic|Cancelled|2052.65|    S105|2026-01-09 07:53:00|user3@example.com|   user3|
|  U1004|      2026-05-21|        

In [13]:
# Fill null values in the status column
df_filled = df.na.fill({"status": "Unknown"})
df_filled.select("status").show(10)

+---------+
|   status|
+---------+
|Cancelled|
|Completed|
|Cancelled|
|  Pending|
|Completed|
|  Unknown|
|Cancelled|
|Cancelled|
|Cancelled|
|Cancelled|
+---------+
only showing top 10 rows


In [14]:
# Use the updated DataFrame
df = df_filled

### Q6 - Count Records by City
Find cities having more than 100 records using `groupBy()` and `count()`.

In [15]:
# Count records for each city having more than 100 records
city_count = df.groupBy("city").count().filter(col("count") > 100)
city_count.show()

+------+-----+
|  city|count|
+------+-----+
| Delhi|  129|
|Jaipur|  220|
+------+-----+



### Q8 - Filter Premium Users
Filter users whose age is between 18 and 30 and whose subscription type is Premium.

In [16]:
# Filter Premium users between age 18 and 30
premium_users = df.filter((col("age").between(18, 30)) & (col("subscription") == "Premium"))
premium_users.show(5)

+-------+----------------+----------------+-----------+------+------+---+------------+---------+-------+--------+-------------------+------------------+--------+
|user_id|transaction_date|product_category|sale_amount|region|  city|age|subscription|   status|  price|store_id|      raw_timestamp|             email|username|
+-------+----------------+----------------+-----------+------+------+---+------------+---------+-------+--------+-------------------+------------------+--------+
|  U1016|      2026-01-17|        Clothing|    3466.87| South|Jaipur| 27|     Premium|Completed|1313.87|    S104|2026-01-10 06:31:00|user16@example.com|  user16|
|  U1017|      2026-01-30|       Groceries|     723.13| North|Jaipur| 27|     Premium|Cancelled|1074.44|    S102|2026-06-24 10:11:00|user17@example.com|  user17|
|  U1019|      2026-05-25|       Furniture|     866.55|  West|Mumbai| 20|     Premium|Completed|1926.24|    S105|2026-04-26 03:27:00|user19@example.com|  user19|
|  U1025|      2026-01-02|  

### Q10 - Modify Schema
Convert the `raw_timestamp` column to TimestampType and rename it to `event_time`.

In [17]:
# Convert raw_timestamp to TimestampType
from pyspark.sql.types import TimestampType

df = df.withColumn("event_time",col("raw_timestamp").cast(TimestampType())).drop("raw_timestamp")
df.printSchema()

root
 |-- user_id: string (nullable = true)
 |-- transaction_date: date (nullable = true)
 |-- product_category: string (nullable = true)
 |-- sale_amount: double (nullable = true)
 |-- region: string (nullable = true)
 |-- city: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- subscription: string (nullable = true)
 |-- status: string (nullable = false)
 |-- price: double (nullable = true)
 |-- store_id: string (nullable = true)
 |-- email: string (nullable = true)
 |-- username: string (nullable = true)
 |-- event_time: timestamp (nullable = true)



### Q12 - Clean Invalid Records
Remove records where the email is null or the username is empty.

In [18]:
# Remove rows with null email or empty username
clean_df = df.filter((col("email").isNotNull()) & (col("username") != ""))

print(df.count())
print(clean_df.count())

480
451


In [19]:
# Use the updated DataFrame
df = clean_df

### Q13 - Aggregate Statistics
Calculate the minimum, maximum, and average values of the `price` column.

In [20]:
# Calculate minimum, maximum and average price
df.agg(
    min("price").alias("Minimum Price"),
    max("price").alias("Maximum Price"),
    avg("price").alias("Average Price")
).show()

+-------------+-------------+------------------+
|Minimum Price|Maximum Price|     Average Price|
+-------------+-------------+------------------+
|        52.18|       2993.6|1539.1979118329464|
+-------------+-------------+------------------+



### Q15 - Final Data Processing Pipeline
Build a simple data processing pipeline by filling missing prices, grouping data by store, and calculating total revenue.

In [21]:
# Fill missing prices and calculate total revenue for each store
pipeline_df = df.na.fill({"price": 0}).groupBy("store_id").agg(sum("price").alias("total_revenue"))
pipeline_df.show()

+--------+------------------+
|store_id|     total_revenue|
+--------+------------------+
|    S105|143593.44999999998|
|    S102|116066.27000000003|
|    S104|120200.56999999998|
|    S101| 148559.5599999999|
|    S103|         134974.45|
+--------+------------------+

